# Statistical Model Comparison (Phase 7)

This notebook answers two questions with formal statistical tests, using the per-fold results produced by Pipelines 1 and 2 (`results/pipeline1/fold_results.csv`, `results/pipeline2/fold_results.csv`):

1. **Friedman test + Nemenyi post-hoc** — are the differences in test RMSE between models (within each dataset) statistically significant?
2. **Wilcoxon signed-rank** — does Multi-output regression have a significant advantage over Single-output regression, per model and dataset (paired over identical seed/fold blocks)?

All tables are saved to `results/statistical_tests/`, and the last cell shows the complete combined results.

Setup: repo imports, pandas display options, load the Pipeline 1 & 2 fold-level results.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if (Path.cwd() / '..' / 'src').resolve().exists() else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

from src.config import get_path
from src.evaluation import friedman_test, nemenyi_posthoc, wilcoxon_pairwise
from src.utils.io import save_table

results_dir = get_path('results_dir')
p1 = pd.read_csv(results_dir / 'pipeline1' / 'fold_results.csv')
p2 = pd.read_csv(results_dir / 'pipeline2' / 'fold_results.csv')
out_dir = results_dir / 'statistical_tests'
print('Pipeline 1 fold rows:', len(p1))
print('Pipeline 2 fold rows:', len(p2))

Pipeline 1 fold rows: 4675
Pipeline 2 fold rows: 825


## Friedman test across models (per dataset)

Blocks = (target, seed, fold); columns = models; values = test RMSE. A significant Friedman result (p < 0.05) means at least one model differs from the rest.

In [2]:
friedman_rows = []
nemenyi_tables = {}
for ds_name, grp in p1.groupby('dataset'):
    matrix = grp.pivot_table(
        index=['target', 'seed', 'fold'], columns='model', values='test_rmse'
    ).dropna()
    res = friedman_test(matrix)
    friedman_rows.append({'dataset': ds_name, **res})
    nemenyi = nemenyi_posthoc(matrix)
    nemenyi.index = nemenyi.columns = matrix.columns
    nemenyi_tables[ds_name] = nemenyi
    save_table(nemenyi, out_dir / f'nemenyi_{ds_name}.csv', index=True)
    print(f'[friedman] {ds_name}: statistic={res["statistic"]:.4f}, p_value={res["p_value"]:.6f}')

friedman_df = pd.DataFrame(friedman_rows)
save_table(friedman_df, out_dir / 'friedman_across_models.csv')
friedman_df

[friedman] Dataset_0136: statistic=1036.5818, p_value=0.000000
[friedman] Dataset_0172: statistic=848.4931, p_value=0.000000
[friedman] Dataset_3772: statistic=614.1324, p_value=0.000000


,dataset,statistic,p_value
0,Dataset_0136,1036.581818,2.457873e-216
1,Dataset_0172,848.493091,7.698972e-176
2,Dataset_3772,614.132364,1.649089e-125


### Nemenyi post-hoc pairwise p-values — Dataset_0136

In [3]:
nemenyi_tables['Dataset_0136']

model,ann,elasticnet,extra_trees,gpr,gradient_boosting,lightgbm,linear_regression,random_forest,ridge,svr_rbf,xgboost
model,,,,,,,,,,,
ann,1.000000e+00,0.000000e+00,1.545324e-09,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,1.978973e-12,0.000000e+00,1.965908e-03,0.000000
elasticnet,0.000000e+00,1.000000e+00,3.248319e-01,0.000000e+00,8.548717e-15,0.000000,9.990581e-01,9.037709e-01,9.968297e-01,2.890348e-05,0.000000
extra_trees,1.545324e-09,3.248319e-01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000,3.836133e-02,9.979340e-01,2.521165e-02,2.775471e-01,0.000000
gpr,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,9.990581e-01,0.127699,8.548717e-15,0.000000e+00,2.509104e-14,0.000000e+00,0.006286
gradient_boosting,0.000000e+00,8.548717e-15,0.000000e+00,9.990581e-01,1.000000e+00,0.009054,6.761147e-12,0.000000e+00,1.773626e-11,0.000000e+00,0.000168
lightgbm,0.000000e+00,0.000000e+00,0.000000e+00,1.276986e-01,9.053835e-03,1.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.997609
linear_regression,0.000000e+00,9.990581e-01,3.836133e-02,8.548717e-15,6.761147e-12,0.000000,1.000000e+00,3.760333e-01,1.000000e+00,2.686488e-07,0.000000
random_forest,1.978973e-12,9.037709e-01,9.979340e-01,0.000000e+00,0.000000e+00,0.000000,3.760333e-01,1.000000e+00,2.959592e-01,2.262135e-02,0.000000
ridge,0.000000e+00,9.968297e-01,2.521165e-02,2.509104e-14,1.773626e-11,0.000000,1.000000e+00,2.959592e-01,1.000000e+00,1.228812e-07,0.000000


### Nemenyi post-hoc pairwise p-values — Dataset_0172

In [4]:
nemenyi_tables['Dataset_0172']

model,ann,elasticnet,extra_trees,gpr,gradient_boosting,lightgbm,linear_regression,random_forest,ridge,svr_rbf,xgboost
model,,,,,,,,,,,
ann,1.000000e+00,2.084430e-07,0.000005,0.000000e+00,0.000000e+00,0.000000,1.083440e-10,1.998401e-15,1.083440e-10,0.000099,0.000000
elasticnet,2.084430e-07,1.000000e+00,0.999978,1.110223e-16,0.000000e+00,0.000000,9.878915e-01,2.832936e-01,9.878915e-01,0.989356,0.000000
extra_trees,5.063312e-06,9.999785e-01,1.000000,0.000000e+00,0.000000e+00,0.000000,8.374984e-01,7.332630e-02,8.374984e-01,0.999970,0.000000
gpr,0.000000e+00,1.110223e-16,0.000000,1.000000e+00,7.129793e-01,0.004115,2.367662e-12,4.007121e-08,2.367662e-12,0.000000,0.000002
gradient_boosting,0.000000e+00,0.000000e+00,0.000000,7.129793e-01,1.000000e+00,0.633919,0.000000e+00,3.985701e-14,0.000000e+00,0.000000,0.012168
lightgbm,0.000000e+00,0.000000e+00,0.000000,4.114716e-03,6.339192e-01,1.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.873804
linear_regression,1.083440e-10,9.878915e-01,0.837498,2.367662e-12,0.000000e+00,0.000000,1.000000e+00,9.506557e-01,1.000000e+00,0.455274,0.000000
random_forest,1.998401e-15,2.832936e-01,0.073326,4.007121e-08,3.985701e-14,0.000000,9.506557e-01,1.000000e+00,9.506557e-01,0.011355,0.000000
ridge,1.083440e-10,9.878915e-01,0.837498,2.367662e-12,0.000000e+00,0.000000,1.000000e+00,9.506557e-01,1.000000e+00,0.455274,0.000000


### Nemenyi post-hoc pairwise p-values — Dataset_3772

In [5]:
nemenyi_tables['Dataset_3772']

model,ann,elasticnet,extra_trees,gpr,gradient_boosting,lightgbm,linear_regression,random_forest,ridge,svr_rbf,xgboost
model,,,,,,,,,,,
ann,1.000000e+00,1.314171e-12,1.998401e-15,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,8.097462e-10,0.000000e+00
elasticnet,1.314171e-12,1.000000e+00,9.993661e-01,9.771073e-13,3.663736e-15,0.000000e+00,1.294701e-01,8.470696e-01,1.173303e-01,9.986042e-01,0.000000e+00
extra_trees,1.998401e-15,9.993661e-01,1.000000e+00,3.657007e-10,2.367662e-12,1.998401e-15,5.928220e-01,9.986042e-01,5.651571e-01,8.470696e-01,0.000000e+00
gpr,0.000000e+00,9.771073e-13,3.657007e-10,1.000000e+00,9.998405e-01,8.975037e-01,9.019026e-05,1.166100e-07,1.087244e-04,7.771561e-16,2.804484e-03
gradient_boosting,0.000000e+00,3.663736e-15,2.367662e-12,9.998405e-01,1.000000e+00,9.986042e-01,2.411054e-06,1.365809e-09,2.985854e-06,0.000000e+00,3.450039e-02
lightgbm,0.000000e+00,0.000000e+00,1.998401e-15,8.975037e-01,9.986042e-01,1.000000e+00,1.181856e-08,2.367662e-12,1.513089e-08,0.000000e+00,3.284449e-01
linear_regression,0.000000e+00,1.294701e-01,5.928220e-01,9.019026e-05,2.411054e-06,1.181856e-08,1.000000e+00,9.825193e-01,1.000000e+00,7.981759e-03,1.110223e-16
random_forest,0.000000e+00,8.470696e-01,9.986042e-01,1.166100e-07,1.365809e-09,2.367662e-12,9.825193e-01,1.000000e+00,9.780115e-01,2.725967e-01,0.000000e+00
ridge,0.000000e+00,1.173303e-01,5.651571e-01,1.087244e-04,2.985854e-06,1.513089e-08,1.000000e+00,9.780115e-01,1.000000e+00,6.910291e-03,1.110223e-16


## Wilcoxon signed-rank — Single-output vs Multi-output

Paired per (dataset, model, seed, fold): compares the mean test RMSE of the Single-output pipeline against the Multi-output pipeline. A significant result (p < 0.05) means the mode matters for that model on that dataset.

In [6]:
single = (
    p1.groupby(['dataset', 'model', 'seed', 'fold'])['test_rmse'].mean().rename('single')
)
multi = (
    p2.groupby(['dataset', 'model', 'seed', 'fold'])['test_rmse'].mean().rename('multi')
)
paired = pd.concat([single, multi], axis=1).dropna().reset_index()

wilcoxon_rows = []
for (ds_name, model), grp in paired.groupby(['dataset', 'model']):
    res = wilcoxon_pairwise(grp['single'], grp['multi'])
    wilcoxon_rows.append(
        {
            'dataset': ds_name,
            'model': model,
            'mean_rmse_single': grp['single'].mean(),
            'mean_rmse_multi': grp['multi'].mean(),
            **res,
        }
    )

wilcoxon_df = pd.DataFrame(wilcoxon_rows)
save_table(wilcoxon_df, out_dir / 'wilcoxon_single_vs_multi.csv')
wilcoxon_df.sort_values('p_value')

,dataset,model,mean_rmse_single,mean_rmse_multi,statistic,p_value
0,Dataset_0136,ann,14.663593,23.446381,0.0,5.960464e-08
1,Dataset_0136,elasticnet,3.806695,5.073338,0.0,5.960464e-08
2,Dataset_0136,extra_trees,4.497586,6.387095,0.0,5.960464e-08
3,Dataset_0136,gpr,1.716033,2.996164,0.0,5.960464e-08
4,Dataset_0136,gradient_boosting,2.325561,3.037171,0.0,5.960464e-08
5,Dataset_0136,lightgbm,1.735093,2.332151,0.0,5.960464e-08
6,Dataset_0136,linear_regression,3.760037,5.015872,0.0,5.960464e-08
7,Dataset_0136,random_forest,4.604464,6.797635,0.0,5.960464e-08
8,Dataset_0136,ridge,3.773790,5.036838,0.0,5.960464e-08
9,Dataset_0136,svr_rbf,10.544121,15.355437,0.0,5.960464e-08


## Final Summary — Complete Statistical Test Results

Friedman + Nemenyi + Wilcoxon tables displayed in full.

In [7]:
print('=' * 90)
print('1) FRIEDMAN TEST ACROSS MODELS (per dataset)')
print('=' * 90)
display(friedman_df)

for ds_name, table in nemenyi_tables.items():
    print()
    print('=' * 90)
    print(f'2) NEMENYI POST-HOC P-VALUES — {ds_name}')
    print('=' * 90)
    display(table)

print()
print('=' * 90)
print('3) WILCOXON SIGNED-RANK — Single-output vs Multi-output')
print('=' * 90)
display(wilcoxon_df)

print()
print('All statistical test tables saved to:', out_dir.resolve())

1) FRIEDMAN TEST ACROSS MODELS (per dataset)


,dataset,statistic,p_value
0,Dataset_0136,1036.581818,2.457873e-216
1,Dataset_0172,848.493091,7.698972e-176
2,Dataset_3772,614.132364,1.649089e-125



2) NEMENYI POST-HOC P-VALUES — Dataset_0136


model,ann,elasticnet,extra_trees,gpr,gradient_boosting,lightgbm,linear_regression,random_forest,ridge,svr_rbf,xgboost
model,,,,,,,,,,,
ann,1.000000e+00,0.000000e+00,1.545324e-09,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,1.978973e-12,0.000000e+00,1.965908e-03,0.000000
elasticnet,0.000000e+00,1.000000e+00,3.248319e-01,0.000000e+00,8.548717e-15,0.000000,9.990581e-01,9.037709e-01,9.968297e-01,2.890348e-05,0.000000
extra_trees,1.545324e-09,3.248319e-01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000,3.836133e-02,9.979340e-01,2.521165e-02,2.775471e-01,0.000000
gpr,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,9.990581e-01,0.127699,8.548717e-15,0.000000e+00,2.509104e-14,0.000000e+00,0.006286
gradient_boosting,0.000000e+00,8.548717e-15,0.000000e+00,9.990581e-01,1.000000e+00,0.009054,6.761147e-12,0.000000e+00,1.773626e-11,0.000000e+00,0.000168
lightgbm,0.000000e+00,0.000000e+00,0.000000e+00,1.276986e-01,9.053835e-03,1.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.997609
linear_regression,0.000000e+00,9.990581e-01,3.836133e-02,8.548717e-15,6.761147e-12,0.000000,1.000000e+00,3.760333e-01,1.000000e+00,2.686488e-07,0.000000
random_forest,1.978973e-12,9.037709e-01,9.979340e-01,0.000000e+00,0.000000e+00,0.000000,3.760333e-01,1.000000e+00,2.959592e-01,2.262135e-02,0.000000
ridge,0.000000e+00,9.968297e-01,2.521165e-02,2.509104e-14,1.773626e-11,0.000000,1.000000e+00,2.959592e-01,1.000000e+00,1.228812e-07,0.000000



2) NEMENYI POST-HOC P-VALUES — Dataset_0172


...


2) NEMENYI POST-HOC P-VALUES — Dataset_3772


model,ann,elasticnet,extra_trees,gpr,gradient_boosting,lightgbm,linear_regression,random_forest,ridge,svr_rbf,xgboost
model,,,,,,,,,,,
ann,1.000000e+00,1.314171e-12,1.998401e-15,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,8.097462e-10,0.000000e+00
elasticnet,1.314171e-12,1.000000e+00,9.993661e-01,9.771073e-13,3.663736e-15,0.000000e+00,1.294701e-01,8.470696e-01,1.173303e-01,9.986042e-01,0.000000e+00
extra_trees,1.998401e-15,9.993661e-01,1.000000e+00,3.657007e-10,2.367662e-12,1.998401e-15,5.928220e-01,9.986042e-01,5.651571e-01,8.470696e-01,0.000000e+00
gpr,0.000000e+00,9.771073e-13,3.657007e-10,1.000000e+00,9.998405e-01,8.975037e-01,9.019026e-05,1.166100e-07,1.087244e-04,7.771561e-16,2.804484e-03
gradient_boosting,0.000000e+00,3.663736e-15,2.367662e-12,9.998405e-01,1.000000e+00,9.986042e-01,2.411054e-06,1.365809e-09,2.985854e-06,0.000000e+00,3.450039e-02
lightgbm,0.000000e+00,0.000000e+00,1.998401e-15,8.975037e-01,9.986042e-01,1.000000e+00,1.181856e-08,2.367662e-12,1.513089e-08,0.000000e+00,3.284449e-01
linear_regression,0.000000e+00,1.294701e-01,5.928220e-01,9.019026e-05,2.411054e-06,1.181856e-08,1.000000e+00,9.825193e-01,1.000000e+00,7.981759e-03,1.110223e-16
random_forest,0.000000e+00,8.470696e-01,9.986042e-01,1.166100e-07,1.365809e-09,2.367662e-12,9.825193e-01,1.000000e+00,9.780115e-01,2.725967e-01,0.000000e+00
ridge,0.000000e+00,1.173303e-01,5.651571e-01,1.087244e-04,2.985854e-06,1.513089e-08,1.000000e+00,9.780115e-01,1.000000e+00,6.910291e-03,1.110223e-16



3) WILCOXON SIGNED-RANK — Single-output vs Multi-output


,dataset,model,mean_rmse_single,mean_rmse_multi,statistic,p_value
0,Dataset_0136,ann,14.663593,23.446381,0.0,5.960464e-08
1,Dataset_0136,elasticnet,3.806695,5.073338,0.0,5.960464e-08
2,Dataset_0136,extra_trees,4.497586,6.387095,0.0,5.960464e-08
3,Dataset_0136,gpr,1.716033,2.996164,0.0,5.960464e-08
4,Dataset_0136,gradient_boosting,2.325561,3.037171,0.0,5.960464e-08
5,Dataset_0136,lightgbm,1.735093,2.332151,0.0,5.960464e-08
6,Dataset_0136,linear_regression,3.760037,5.015872,0.0,5.960464e-08
7,Dataset_0136,random_forest,4.604464,6.797635,0.0,5.960464e-08
8,Dataset_0136,ridge,3.773790,5.036838,0.0,5.960464e-08
9,Dataset_0136,svr_rbf,10.544121,15.355437,0.0,5.960464e-08



All statistical test tables saved to: C:\Users\mohammadhosein\Desktop\DeepLearning-miniProject\results\statistical_tests
